In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
%pip install xgboost lightgbm catboost

������� ������� ��������: 1251
  Using cached lightgbm-4.7.0-py3-none-win_amd64.whl.metadata (18 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached plotly-7.1.0-py3-none-any.whl.metadata (8.8 kB)
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
    --------------------------------------- 2.4/101.7 MB 19.2 MB/s eta 0:00:06
   -- ------------------------------------- 7.1/101.7 MB 20.8 MB/s eta 0:00:05
   ---- ----------------------------------- 11.3/101.7 MB 20.7 MB/s eta 0:00:05
   ------ --------------------------------- 16.0/101.7 MB 21.0 MB/s eta 0:00:05
   -------- ------------------------------- 20.7/101.7 MB 21.1 MB/s eta 0:00:04
   ---------- ----------------------------- 25.4/101.7 MB 21.5 MB/s eta 0:00:04
   ----------- ---------------------------- 29.6/101.7 MB 21.4 MB/s eta 0:00:04
   ------------ --------------------------- 32.2/101.7 MB 19.9 MB/s eta 0:00:04
   ------------- -------------------------- 34.6/101.7 MB 18.8 M

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [4]:
import numpy as np
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost, lightgbm, catboost

from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import ParameterGrid
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import config as cnf 
import utils as utl

In [13]:
df_train                = pd.read_csv(cnf.path['train_path'])
df_test                 = pd.read_csv(cnf.path['test_path'])
df_gender_submission    = pd.read_csv(cnf.path['gender_submission_path'])

X, y = utl.predobr(df_train, df_test, df_gender_submission)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=cnf.random_state)


array_train_accuracy = []
array_test_accuracy = []

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    current_X_train = X.iloc[train_index]
    current_y_train = y.iloc[train_index]
    current_X_test  = X.iloc[test_index]
    current_y_test  = y.iloc[test_index]
    
    model = catboost.CatBoostClassifier()

    model.fit(current_X_train, current_y_train)
    
    train_accuracy = model.score(current_X_train, current_y_train)
    test_accuracy = model.score(current_X_test, current_y_test)
    
    array_train_accuracy.append(train_accuracy)
    array_test_accuracy.append(test_accuracy)
    
mean_train_accuracy, mean_train_std, mean_test_accuracy, mean_test_std = utl.mean_accuracy(array_train_accuracy, array_test_accuracy)
print("Train_accuracy: ", mean_train_accuracy, "Test_accuracy: ", mean_test_accuracy)

Learning rate set to 0.008911
0:	learn: 0.6874411	total: 1.55ms	remaining: 1.55s
1:	learn: 0.6812816	total: 3.18ms	remaining: 1.58s
2:	learn: 0.6746578	total: 4.89ms	remaining: 1.63s
3:	learn: 0.6692873	total: 6.29ms	remaining: 1.57s
4:	learn: 0.6638023	total: 7.45ms	remaining: 1.48s
5:	learn: 0.6583428	total: 9ms	remaining: 1.49s
6:	learn: 0.6531961	total: 10.4ms	remaining: 1.48s
7:	learn: 0.6475685	total: 11.7ms	remaining: 1.45s
8:	learn: 0.6427701	total: 13ms	remaining: 1.43s
9:	learn: 0.6377936	total: 14.7ms	remaining: 1.46s
10:	learn: 0.6334269	total: 16.4ms	remaining: 1.47s
11:	learn: 0.6283290	total: 17.8ms	remaining: 1.47s
12:	learn: 0.6232103	total: 19.1ms	remaining: 1.45s
13:	learn: 0.6182957	total: 20.5ms	remaining: 1.44s
14:	learn: 0.6134266	total: 22.2ms	remaining: 1.46s
15:	learn: 0.6095145	total: 23.6ms	remaining: 1.45s
16:	learn: 0.6054411	total: 25.1ms	remaining: 1.45s
17:	learn: 0.6011742	total: 26.4ms	remaining: 1.44s
18:	learn: 0.5969484	total: 27.6ms	remaining: 1.4

In [10]:
df_train                = pd.read_csv(cnf.path['train_path'])
df_test                 = pd.read_csv(cnf.path['test_path'])
df_gender_submission    = pd.read_csv(cnf.path['gender_submission_path'])

X, y = utl.predobr(df_train, df_test, df_gender_submission)

combo_param = utl.parametrs(cnf.RandomForestParam) + utl.parametrs(cnf.LogRegParam)
combo_param += utl.parametrs(cnf.KNNParam)
combo_param += utl.parametrs(cnf.DecisionTreeParam)
combo_param += utl.parametrs(cnf.CatBoostParam)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=cnf.random_state)
results = []

for combo in combo_param:
    
    params = combo.copy()
    name_model = params.pop('name_model')
    scaler = params.pop('scaler')
    
    print("Model: ", name_model, "Params: ", params)
    
    model = cnf.models[name_model](**params)
    mean_train_accuracy, mean_train_std, mean_test_accuracy, mean_test_std = utl.skf_function(model, X, y, skf, scaler)

    results.append({
        **combo
        , 'train_accuracy'  :   mean_train_accuracy
        , 'train_std'       :   mean_train_std
        , 'test_accuracy'   :   mean_test_accuracy
        , 'test_std'        :   mean_test_std
    })

Model:  RandomForest Params:  {'max_depth': 8, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 42}
Train_accuracy:  0.9158229194572701 Test_accuracy:  0.8361245370660976
Model:  RandomForest Params:  {'max_depth': 8, 'min_samples_leaf': 2, 'n_estimators': 100, 'random_state': 42}
Train_accuracy:  0.8947795357486171 Test_accuracy:  0.8372355784319879
Model:  RandomForest Params:  {'max_depth': 8, 'min_samples_leaf': 3, 'n_estimators': 100, 'random_state': 42}
Train_accuracy:  0.8835581574924752 Test_accuracy:  0.8349946644906158
Model:  RandomForest Params:  {'max_depth': 9, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 42}
Train_accuracy:  0.9326587295333848 Test_accuracy:  0.8428535559600778
Model:  RandomForest Params:  {'max_depth': 9, 'min_samples_leaf': 2, 'n_estimators': 100, 'random_state': 42}
Train_accuracy:  0.9051617630836629 Test_accuracy:  0.8327537505492437
Model:  RandomForest Params:  {'max_depth': 9, 'min_samples_leaf': 3, 'n_estimators': 100,

TypeError: CatBoostClassifier.score() got an unexpected keyword argument 'sample_weight'

### Вывод

In [9]:
results = pd.DataFrame(results)
results['gap'] = results['train_accuracy'] - results['test_accuracy']

# Порядок колонок: сначала модель, потом метрики, потом гиперпараметры
metrics = ['test_accuracy', 'test_std', 'train_accuracy', 'train_std', 'gap']
service = ['name_model', 'scaler', 'random_state']
params = [c for c in results.columns if c not in metrics + service]

results = results[['name_model'] + metrics + params + ['scaler', 'random_state']]

# Сортировка: лучшие по test_accuracy сверху
results = results.sort_values('test_accuracy', ascending=False).reset_index(drop=True)
results.head(50)

,name_model,test_accuracy,test_std,train_accuracy,train_std,gap,max_depth,min_samples_leaf,n_estimators,n_neighbors,scaler,random_state
0,RandomForest,0.842854,0.012525,0.932659,0.004454,0.089805,9.0,1.0,100.0,NaN,False,42.0
1,RandomForest,0.839483,0.019818,0.888046,0.004415,0.048563,9.0,3.0,100.0,NaN,False,42.0
2,RandomForest,0.837236,0.022066,0.894780,0.004647,0.057544,8.0,2.0,100.0,NaN,False,42.0
3,RandomForest,0.836125,0.014097,0.915823,0.006486,0.079698,8.0,1.0,100.0,NaN,False,42.0
4,RandomForest,0.834995,0.020443,0.883558,0.005809,0.048563,8.0,3.0,100.0,NaN,False,42.0
5,RandomForest,0.832754,0.018059,0.905162,0.002320,0.072408,9.0,2.0,100.0,NaN,False,42.0
6,DecisionTree,0.829383,0.015938,0.879064,0.008909,0.049681,8.0,3.0,NaN,NaN,False,42.0
7,DecisionTree,0.827142,0.018064,0.887763,0.006209,0.060621,9.0,3.0,NaN,NaN,False,42.0
8,DecisionTree,0.823784,0.012735,0.883555,0.006451,0.059771,8.0,2.0,NaN,NaN,False,42.0
9,DecisionTree,0.821549,0.020218,0.902074,0.005327,0.080525,8.0,1.0,NaN,NaN,False,42.0
